<a href="https://colab.research.google.com/github/vencov/FAV_course/blob/main/TutIHC03/TutIHCbiophysical_original.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Biophysical inner-hair-cell (IHC) and auditory-nerve (AN) model

Python/Jupyter version **MAP** (the Matlab Auditory Periphery) model by Ray Meddis follows Shamma et al. (1986) and Sumner et al. (2002) for IHC simulation.

A pure tone with raised-cosine onset/offset ramps is used as **basilar-membrane (BM) displacement** at one place
along the cochlea. It is passed through the following stages:

| Stage | Function | Input → Output |
|---|---|---|
| 1. IHC receptor potential | `IHC_RP_Shamma` | BM displacement [m] → IHC membrane potential *V* [V] |
| 2. Pre-synapse | `IHCpreSynapse` | *V* → vesicle release rate *k(t)* [1/s] for HSR / MSR / LSR synapses |
| 3. Synapse and AN fiber | `AN_IHCsynapse` | release rate → AN spikes (200 fibers) and spike probability |
| 4. Analysis | `UTIL_PSTHmaker`, `UTIL_periodHistogram`, `UTIL_vectorStrength` | spikes → PSTH, period histogram, vector strength |

HSR / MSR / LSR = high / medium / low spontaneous-rate auditory-nerve fibers.

**References**
- Shamma, Chadwick, Wilbur, Morrish, Rinzel (1986). *A biophysical model of cochlear processing: intensity dependence of pure tone responses.* JASA 80(1), 133–145.
- Sumner, Lopez-Poveda, O'Mard, Meddis (2002). *A revised model of the inner-hair cell and auditory-nerve complex.* JASA 111(5).

**Original code by** Almudena Eustaquio-Martin (Salamanca), Ray Meddis (Essex), modified by Vaclav Vencovsky (CTU Prague).

In [ ]:
import warnings
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
from matplotlib.ticker import ScalarFormatter, NullFormatter

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 3.5)
plt.rcParams["axes.grid"] = True

## 1. IHC receptor potential (`IHC_RP_Shamma`)

The BM displacement *w* is converted into the IHC membrane potential in three steps.

**a) BM → stereocilia displacement.** First-order high-pass filter with gain `Ccilia` and time constant `tau_c`
(Shamma eq. 1). The original labels the gain as dB but applies it as a linear factor, which is kept here.

$$\tau_c \frac{du}{dt} + u = \tau_c\, C_{cilia}\, \frac{dw}{dt}$$

**b) Displacement → apical conductance.** A Boltzmann function gives the mechanically gated conductance.
The second-order form (Sumner, default) is

$$G_{mu}(u) = \frac{G_{max}}{1 + e^{(u_0-u)/s_0}\left(1+e^{(u_1-u)/s_1}\right)}, \qquad G_u = G_{mu} + G_a$$

where the leakage conductance $G_a = G_0 - G_{mu}(0)$. A first-order form (Shamma) is available via `BoltzmanOrder=1`.

**c) Conductance → membrane potential.** An RC circuit with total capacitance $C_{ab}$, endocochlear potential $E_t$
and potassium conductance $G_k$ (discretised implicitly in *V*, as in the original code):

$$C_{ab}\frac{dV}{dt} = G_u(E_t - V) + G_k(E_k' - V), \qquad E_k' = E_k + q\,E_t$$

## 2. Pre-synapse: calcium dynamics and vesicle release (`IHCpreSynapse`)

The membrane potential *V* opens voltage-gated calcium channels. The fraction of open channels follows a first-order
process towards a Boltzmann steady state; calcium then accumulates and is cleared with a time constant $\tau_{Ca}$
(Sumner et al., 2002, eq. 7):

$$m_\infty = \frac{1}{1+e^{-\gamma V}/\beta}, \quad \tau_M\frac{dm}{dt} = m_\infty - m, \quad I_{Ca} = G_{Ca}\,m^3 (V - E_{Ca}), \quad \frac{d[Ca]}{dt} = I_{Ca} - \frac{[Ca]}{\tau_{Ca}}$$

Vesicle release rate is a power law of the calcium concentration: $k(t) = z\,[Ca]^{3}$.

Three synapse types are computed in parallel, differing only in $\tau_{Ca}$ (HSR 0.35 ms, MSR 0.15 ms, LSR 0.075 ms).
The result has shape *(time × channels × 3)*. `method.kt0` holds the release rate at rest for each type
and is needed to initialise the synapse in the next stage.

> The `Cathr_*` thresholds and `GmaxCa_*` values in the default parameters belong to an alternative *influx* model
> that is commented out in the original code. They are kept in the dictionary but are not used.

## 3. Synapse and auditory-nerve fiber (`AN_IHCsynapse`)

Vesicle circulation is described by three stores (Meddis, 2006 / MAP):

- **q** – vesicles available for release at the synapse (maximum `M`),
- **c** – transmitter in the synaptic cleft,
- **w** – transmitter in the reprocessing store.

$$\frac{dq}{dt} = y(M-q) - k\,q + x\,w, \qquad \frac{dc}{dt} = k\,q - (l+r)\,c, \qquad \frac{dw}{dt} = r\,c - x\,w$$

where *k(t)* is the release rate from the previous stage. The stores start in their steady state for the resting rate `kt0`.

Two versions are computed:

1. **Spikes (stochastic).** `numFibers` (200) independent fibers. In every time step the number of vesicles that
   are replenished, released and reprocessed is drawn randomly (`AN_N1`). Each release event is a spike, followed by a
   refractory period: an absolute part of 0.75 ms plus a random relative part of up to the same length.
2. **Probability (deterministic).** The same equations on the mean contents, giving the expected spike rate without refractoriness.

For `mode='spikes'` the function returns the *(fibers × time)* spike matrix as `synapseOutput` and the deterministic
rate as `ANprobability`. With `mode='probability'` the rate is returned in both outputs.

In [ ]:
def default_ihc_params():
    """Default IHC receptor-potential parameters (Sumner et al., 2002)."""
    return dict(
        Ccilia=16,         # cilia/BM coupling gain (used as a linear factor)
        tau_c=2.13e-3,     # cilia/BM time constant [s]
        BoltzmanOrder=2,   # 1 = first-order (Shamma), 2 = second-order (Sumner)
        Gmax=8e-9,         # max mechanical conductance [S]
        Gk=1.8e-8,         # potassium conductance [S]
        G0=1.974e-9,       # resting conductance Gu(u=0) [S]
        # first-order Boltzmann (only if BoltzmanOrder == 1)
        Gamma=10e6, Beta=0.25,
        # second-order Boltzmann (only if BoltzmanOrder == 2)
        s0=85e-9, u0=7e-9, s1=5e-9, u1=7e-9,
        Et=100e-3,         # endocochlear potential [V]
        Ek=-70.45e-3,      # K+ reversal potential [V]
        q=0.04,            # reversal potential correction Rp/(Rp+Rt)
        Cab=6e-12,         # total capacitance Ca+Cb [F]
    )


def IHC_RP_Shamma(w, method, params=None):
    """Shamma's IHC receptor potential model.

    w      : BM displacement, shape (channels, time) or (time,) [m]
    method : namespace with .dt (sample period); .restingV is set here
    params : dict as returned by default_ihc_params() (defaults used if None)

    Returns V (channels, time) [V], the updated method, and an (empty) info dict.
    """
    p = default_ihc_params() if params is None else params
    dt = method.dt
    w = np.atleast_2d(np.asarray(w, dtype=float))

    # --- BM displacement -> cilia displacement (high-pass filter) ----------
    a = [1.0, -p["tau_c"] / (p["tau_c"] + dt)]
    b0 = p["tau_c"] * p["Ccilia"] / (p["tau_c"] + dt)
    u = lfilter([b0, -b0], a, w, axis=1)

    Ekp = p["Ek"] + p["Et"] * p["q"]

    # --- cilia displacement -> mechanical conductance (Boltzmann) ----------
    if p["BoltzmanOrder"] == 1:
        Pc0 = 1 + (1 / p["Beta"]) * np.exp(-p["Gamma"] * 0)
        Pc = 1 + (1 / p["Beta"]) * np.exp(-p["Gamma"] * u)
    elif p["BoltzmanOrder"] == 2:
        Pc0 = 1 + np.exp(p["u0"] / p["s0"]) * (1 + np.exp(p["u1"] / p["s1"]))
        Pc = 1 + np.exp((p["u0"] - u) / p["s0"]) * (1 + np.exp((p["u1"] - u) / p["s1"]))
    else:
        raise ValueError("BoltzmanOrder must be 1 or 2")

    Gmu0 = p["Gmax"] / Pc0          # resting mechanical conductance
    Gmu = p["Gmax"] / Pc            # Gmu(t)
    Ga = p["G0"] - Gmu0             # apical leakage conductance
    if Ga < 0:
        warnings.warn(f"IHC_RP_Shamma: apical leakage conductance (Ga = {Ga:g}) < 0; "
                      "resting conductance is too low")
    Gu = Gmu + Ga                   # total membrane conductance Gu(t)

    # --- conductance -> membrane potential (with capacitance) --------------
    V0 = (p["Et"] * p["G0"] + p["Gk"] * Ekp) / (p["G0"] + p["Gk"])   # resting potential
    method.restingV = V0

    Cdt = p["Cab"] / dt
    V = np.zeros_like(u)
    V[:, 0] = V0
    for i in range(1, u.shape[1]):
        V[:, i] = (p["Et"] * Gu[:, i - 1] + Ekp * p["Gk"] + Cdt * V[:, i - 1]) \
                  / (Cdt + Gu[:, i - 1] + p["Gk"])

    return V, method, {}


def default_presynapse_params():
    return dict(
        ECa=0.066,          # calcium reversal potential [V]
        beta=400, gamma=130,  # Boltzmann parameters
        tauM=1e-4,          # calcium channel time constant [s]
        tauCa=1e-4,
        tauCa_HSR=3.5e-4, tauCa_MSR=1.5e-4, tauCa_LSR=0.75e-4,   # calcium clearance time constants [s]
        power=3, z=2e42,
        GmaxCa=8e-9,
        # unused (alternative influx model, commented out in the original)
        Cathr_HSR=0, Cathr_MSR=3.2e-14, Cathr_LSR=1.4e-11,
        GmaxCa_HSR=14e-9, GmaxCa_MSR=10e-9, GmaxCa_LSR=9e-9,
    )


def IHCpreSynapse(V, method, params=None):
    """Convert IHC potential to vesicle release rates.

    V      : IHC potential, shape (time, channels) [V]
    method : namespace with .dt and .restingV; .kt0, .tauCa, .numSynapses are set here
    params : dict as returned by default_presynapse_params()

    Returns vesicleReleaseRate with shape (time, channels, synapse types) in events/s
    (rate for a single available vesicle) and the updated method.
    """
    p = default_presynapse_params() if params is None else params
    V = np.asarray(V, dtype=float)
    dt, restingV = method.dt, method.restingV

    method.tauCa = p["tauCa"]
    ECa, gamma, beta, tauM = p["ECa"], p["gamma"], p["beta"], p["tauM"]
    GmaxCa, z, power = p["GmaxCa"], p["z"], p["power"]

    # one clearance time constant per synapse type present in params (HSR, MSR, LSR)
    tauCa = np.array([p[k] for k in ("tauCa_HSR", "tauCa_MSR", "tauCa_LSR") if k in p])
    nTypes = tauCa.size

    numDataPoints, numChannels = V.shape
    method.numSynapses = np.size(p["tauCa"]) * numChannels
    # (the MATLAB code also replicates V once per element of params.tauCa; with the
    #  scalar default this is a no-op, so it is left out)

    # steady-state fraction of open channels
    mICaINF = 1.0 / (1.0 + np.exp(-gamma * V) / beta)

    # resting values
    mICa = np.zeros_like(V)
    mICa[0, :] = 1.0 / (1.0 + np.exp(-gamma * restingV) / beta)
    ICa0 = GmaxCa * mICa[0, :] ** 3 * (restingV - ECa)           # (channels,)
    restingCa = ICa0[:, None] * tauCa[None, :]                    # (channels, types); negative
    method.kt0 = -z * restingCa[0, :] ** power                    # release rate at rest, (types,)

    # fraction of open calcium channels
    for i in range(1, numDataPoints):
        mICa[i, :] = mICa[i - 1, :] + (mICaINF[i, :] - mICa[i - 1, :]) * dt / tauM

    # calcium current and (negative) accumulated calcium
    mICa3 = np.repeat(mICa[:, :, None], nTypes, axis=2)
    ICa = GmaxCa * mICa3 ** 3 * (V[:, :, None] - ECa)
    Ca = np.zeros_like(ICa)
    Ca[0] = restingCa
    for i in range(1, numDataPoints):
        Ca[i] = Ca[i - 1] + ICa[i] * dt - Ca[i - 1] * dt / tauCa

    Ca = -Ca                                   # treat calcium as a positive quantity
    vesicleReleaseRate = z * Ca ** power       # events/s, independent of dt
    return vesicleReleaseRate, method

def AN_N1(x, p, rng):
    """Random test for each possible event.

    x : number of eligible vesicles, p : probability per vesicle.
    Returns 0/1 (as float) for whether at least one event occurred, per element.
    """
    prob = 1.0 - (1.0 - p) ** x
    return (prob > rng.random(np.shape(prob))).astype(float)


def AN_IHCsynapse(kdt, method, params=None, rng=None):
    """Convert IHC vesicle release rates to auditory-nerve activity.

    kdt    : release rates, shape (channels, time) [events/s]
    method : namespace with .dt and .kt0 (scalar release rate at rest)
    params : dict (see defaults below); rng : numpy Generator for the spikes

    Returns (synapseOutput, method, ANprobability).
    """
    if params is None:
        params = dict(
            mode="spikes",              # 'spikes' or 'probability'
            numFibers=200,              # number of fibers per channel (spikes only)
            M=10,                       # max number of available vesicles
            y=3,                        # slow replenishment rate
            l=2580,                     # rate of loss from cleft
            x=30,                       # rate of release from the reprocessing store
            r=6580,                     # rate of reuptake from the cleft
            refractory_period=0.00075,  # [s]
            returnSynapseContents=0,    # 1: ANprobability = [q; c; w; kdt] over time
        )
    rng = np.random.default_rng() if rng is None else rng
    mode = params["mode"]
    y, l, x, r, M = params["y"], params["l"], params["x"], params["r"], int(np.floor(params["M"] + 0.5))
    returnContents = params.get("returnSynapseContents", 0)

    dt = method.dt
    kdt = np.atleast_2d(np.asarray(kdt, dtype=float)) * dt     # events/s -> events/sample
    numChannels, numDataPoints = kdt.shape
    kt0 = float(np.squeeze(method.kt0))                        # release rate at rest

    def initial_stores(n):
        """Steady-state values of c, q, w for rest rate kt0 (vectors of length n)."""
        c = kt0 * y * M / (y * (l + r) + kt0 * l)
        q = c * (l + r) / kt0
        w = c * r / x
        return np.full(n, c), np.full(n, q), np.full(n, w)

    synapseOutput = None
    ANprobability = None

    # ---------------------------------------------------------------- spikes
    if mode == "spikes":
        nFib = params["numFibers"]
        n = numChannels * nFib                      # fibers are ordered by channel
        ydt, ldt, xdt, rdt = y * dt, l * dt, x * dt, r * dt
        c, q, w = initial_stores(n)

        synapseOutput = np.zeros((n, numDataPoints), dtype=bool)
        for i in range(numDataPoints):
            k = np.repeat(kdt[:, i], nFib)          # release probability per fiber
            q = np.floor(q + 0.5)                   # vesicle counts must be integers
            M_q = np.maximum(M - q, 0)
            replenish = AN_N1(M_q, ydt, rng)
            ejected = AN_N1(q, k, rng)
            reprocessed = AN_N1(np.floor(w), xdt, rng)
            reuptakeandlost = (rdt + ldt) * c
            reuptake = rdt * c

            q = q + replenish - ejected + reprocessed
            c = c + ejected - reuptakeandlost
            w = w + reuptake - reprocessed
            synapseOutput[:, i] = ejected > 0       # release event = spike

        # Remove spikes that fall in the refractory period of a preceding spike:
        # absolute period + a random relative period (between 1x and 2x absolute).
        nAbs = int(np.floor(params["refractory_period"] / dt + 0.5))
        for i in range(numDataPoints - 2 * nAbs):
            for k_idx in np.flatnonzero(synapseOutput[:, i]):
                nRef = nAbs + int(np.floor(rng.random() * nAbs + 0.5))
                synapseOutput[k_idx, i + 1:i + nRef + 1] = False

        method.ANresponseType = "spikes"
        method.numANfibers = nFib
    else:
        method.ANresponseType = mode
        method.numANfibers = 1

    method.AN_IHCsynapsedt = dt

    # ----------------------------------------------------------- probability
    # (always computed; no refractory effect, gives a general impression of the rate)
    ydt, ldt, xdt, rdt = y * dt, l * dt, x * dt, r * dt
    c, q, w = initial_stores(numChannels)
    ANprobability = np.zeros((numChannels, numDataPoints))
    if returnContents:
        qt = np.zeros((numChannels, numDataPoints))
        ct = np.zeros_like(qt)
        wt = np.zeros_like(qt)

    for i in range(numDataPoints):
        replenish = (M - q) * ydt
        ejected = q * kdt[:, i]
        reprocessed = w * xdt
        ANprobability[:, i] = ejected
        reuptakeandlost = (rdt + ldt) * c
        reuptake = rdt * c

        q = q + replenish - ejected + reprocessed
        c = c + ejected - reuptakeandlost
        w = w + reuptake - reprocessed
        if returnContents:
            qt[:, i], ct[:, i], wt[:, i] = q, c, w

    ANprobability = ANprobability / dt              # probability per sample -> rate [1/s]

    if mode == "probability":
        synapseOutput = ANprobability
    if returnContents:
        ANprobability = np.vstack([qt, ct, wt, kdt])

    return synapseOutput, method, ANprobability

## 4. Analysis utilities

- **`UTIL_PSTHmaker`** – sums spike counts into bins of width `PSTHbinWidth` (post-stimulus time histogram).
- **`UTIL_periodHistogram`** – folds the spike train on the stimulus period (1/frequency) and sums across periods.
  The period must be a whole number of samples.
- **`UTIL_vectorStrength`** – synchronization index of a period histogram (Johnson, 1980): the length of the mean vector
  of the spike phases, from 0 (no phase locking) to 1 (perfect phase locking).

### Task: implement the vector strength

The period histogram `PH` counts the spikes that fall into each of *K* bins of one stimulus period (one row per channel).
Bin *k* corresponds to the phase $\varphi_k = 2\pi k/K$. The vector strength (synchronization index, Johnson 1980)
treats every spike as a unit vector at its phase and takes the length of the mean vector:

$$VS = \sqrt{\left(\frac{1}{N}\sum_{k=1}^{K} PH_k \cos\varphi_k\right)^2 + \left(\frac{1}{N}\sum_{k=1}^{K} PH_k \sin\varphi_k\right)^2}, \qquad N = \sum_{k=1}^{K} PH_k$$

Complete the function below. The phase `phase` ($\varphi_k$) and the number of spikes `N` are already computed for you.


Requirements:
- vectorised NumPy, no loop over bins,
- it must work for several channels at once (rows of `PH`) and return one value per channel,
- hint: `np.sum(..., axis=1)`.


In [ ]:
def UTIL_PSTHmaker(inputData, dt, PSTHbinWidth):
    """Accumulate values (channels x time) into bins of width PSTHbinWidth.

    Returns the binned matrix (sum within each bin) and dt (returned unchanged, as in the original).
    """
    inputData = np.atleast_2d(inputData)
    numChannels, numDataPoints = inputData.shape
    dataPointsPerBin = int(np.floor(PSTHbinWidth / dt + 0.5))
    if dataPointsPerBin <= 1:                      # too few data points per bin
        return inputData, dt

    numBins = numDataPoints // dataPointsPerBin
    used = numBins * dataPointsPerBin              # drop the unusable tail
    PSTH = inputData[:, :used].reshape(numChannels, numBins, dataPointsPerBin).sum(axis=2)
    return PSTH, dt


def UTIL_periodHistogram(A, dt, frequency):
    """Convert a (channels x time) sequence into a period histogram.

    Returns PH (channels x points per period) and binTimes (for plotting).
    """
    A = np.atleast_2d(A)
    period = 1.0 / frequency
    numChannels, nPts = A.shape

    pointsPerPeriod = int(np.floor(period / dt + 0.5))
    NcompletePeriods = nPts // pointsPerPeriod
    totalPointsUsed = NcompletePeriods * pointsPerPeriod

    # the period must be a whole number of samples
    aliasing = NcompletePeriods * (period / dt - pointsPerPeriod)
    if aliasing > 0.1:
        raise ValueError("UTIL_periodHistogram: irregular period length")
    if NcompletePeriods < 1:
        raise ValueError("UTIL_periodHistogram: too few datapoints")

    A = A[:, :totalPointsUsed].reshape(numChannels, NcompletePeriods, pointsPerPeriod)
    PH = A.sum(axis=1)
    binTimes = dt * np.arange(1, pointsPerPeriod + 1)
    return PH, binTimes


def UTIL_vectorStrength(PH):
    """Vector strength (synchronization index, Johnson 1980) per channel of a period histogram."""
    PH = np.atleast_2d(PH)
    K = PH.shape[1]
    phase = 2 * np.pi * np.arange(1, K + 1) / K
    N = PH.sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):   # NaN if there are no spikes
        # TODO (HOMEWORK)
        # VS =
    return VS

### Test your VS calculation function

In [ ]:
# Test of UTIL_vectorStrength on histograms with a known result
K = 20
phase = 2 * np.pi * np.arange(K) / K

PH_one  = np.zeros(K); PH_one[7] = 100     # all spikes in one bin      -> r = 1
PH_flat = np.full(K, 5.0)                  # same count in every bin    -> r = 0
PH_cos  = 1 + np.cos(phase)                # PH_k = 1 + cos(phi_k)      -> r = 0.5

# three channels in one call (rows = channels), this also tests the vectorization
PH_test  = np.vstack([PH_one, PH_flat, PH_cos])
expected = np.array([1.0, 0.0, 0.5])

VS_test = UTIL_vectorStrength(PH_test)
assert VS_test is not None, "UTIL_vectorStrength returns None - did you replace None in the line 'VS ='?"
VS_test = np.asarray(VS_test, dtype=float)
assert VS_test.shape == (3,), f"Expected one value per channel, shape (3,), got {VS_test.shape}"

names = ["all spikes in one bin", "uniform histogram", "1 + cos(phase)"]
for name, got, exp in zip(names, VS_test, expected):
    status = "OK" if np.isclose(got, exp, atol=1e-6) else "WRONG"
    print(f"{name:24s} expected {exp:.3f}   got {got:.6f}   {status}")
assert np.allclose(VS_test, expected, atol=1e-6), "At least one test failed - check your equation."

# the result must not depend on the total number of spikes
assert np.allclose(UTIL_vectorStrength(10 * PH_test), expected, atol=1e-6), \
    "VS must not change when all counts are multiplied by a constant (did you divide by N?)"

# single channel given as a 1 x K array
print("Single channel:", UTIL_vectorStrength(PH_cos[None, :]))

# histogram without any spike: what do you get and why?
with np.errstate(invalid="ignore", divide="ignore"):
    print("Empty histogram:", UTIL_vectorStrength(np.zeros((1, K))))

print("\nAll tests passed.")

## 5. Run the model

### Stimulus

A 2 kHz tone (100 ms, amplitude 10 nm) represents the BM displacement at one cochlear place. It is shaped by 5 ms
raised-cosine onset and offset ramps and padded with 10 ms of silence before and 30 ms after.
Change `freq` and `Amp` to explore other stimulus levels; the sampling frequency must make the stimulus period a
whole number of samples (needed for the period histogram).

### IHC receptor potential
BM displacement → cilia displacement → apical K⁺ conductance → membrane potential.

### Pre-synapse and vesicle release
The membrane potential opens calcium channels, and the calcium influx drives the release of neurotransmitter vesicles.
The output contains one release-rate trace per synapse type (HSR, MSR, LSR).

### Synapse and AN fiber
Neurotransmitter circulation and spike generation are run separately for each fiber type, each starting from its own
resting release rate `kt0`. The spike matrices have shape *(200 fibers × time)*. This cell takes a few seconds because of the
sample-by-sample stochastic simulation.

### Post-stimulus time histograms (PSTH)
Spikes are averaged over the 200 fibers and summed in bins of 10 samples (0.1 ms). Dividing by the bin width gives
the mean firing rate of a single fiber in spikes/s.

In [ ]:
SEED = 1234                      # seed for the spike generation
rng = np.random.default_rng(SEED)

fs = 100e3                       # sampling frequency [Hz]


def make_stimulus(freq, Amp, fs, Tdur=100e-3, Rdur=5e-3, pre=10e-3, post=30e-3):
    """Tone with raised-cosine onset/offset ramps, padded with silence.

    Tdur: tone duration, Rdur: ramp duration, pre/post: silence before/after [s].
    Returns the BM displacement [m] and the time axis [s].
    """
    tx = np.arange(int(round(Tdur * fs)) + 1) / fs
    sig = Amp * np.sin(2 * np.pi * freq * tx)            # BM displacement at one point

    # raised-cosine ramps
    x = np.arange(int(round(Rdur * fs)) + 1) / fs
    x = np.pi * x / Rdur
    rampUp = (1 + np.cos(x + np.pi)) / 2
    rampDown = rampUp[::-1]
    wholeramp = np.concatenate([rampUp, np.ones(len(sig) - 2 * len(x)), rampDown])

    sig = np.concatenate([np.zeros(int(round(pre * fs))), sig * wholeramp,
                          np.zeros(int(round(post * fs)))])
    return sig, np.arange(len(sig)) / fs


freq = 2000                      # tone frequency [Hz]
Amp = 1e-8                       # amplitude [m]
sigIn, txL = make_stimulus(freq, Amp, fs)

fig, ax = plt.subplots()
ax.plot(txL * 1e3, sigIn * 1e9)
ax.set(xlabel="Time (ms)", ylabel="BM displacement (nm)", title="Input stimulus")
plt.show()

method = SimpleNamespace(dt=1 / fs)
V, method, info = IHC_RP_Shamma(sigIn, method)

fig, ax = plt.subplots()
ax.plot(txL * 1e3, V[0])
ax.set(xlabel="Time (ms)", ylabel="Membrane potential (V)", title="IHC receptor potential")
plt.show()

# V has shape (channels, time); IHCpreSynapse expects (time, channels)
vesicleReleaseRate, method = IHCpreSynapse(V.T, method)

vesicle = {name: vesicleReleaseRate[:, :, k] for k, name in enumerate(["HSR", "MSR", "LSR"])}
kt0 = {name: method.kt0[k] for k, name in enumerate(["HSR", "MSR", "LSR"])}   # release rate at rest

# optional: look at the release rate
# plt.plot(txL * 1e3, vesicle["HSR"][:, 0]); plt.show()

synapseOutput, ANprobability = {}, {}
for name in ["HSR", "MSR", "LSR"]:
    method.kt0 = kt0[name]                     # AN_IHCsynapse reads the resting rate from method
    synapseOutput[name], method, ANprobability[name] = AN_IHCsynapse(vesicle[name].T, method, rng=rng)
    print(f"{name}: {int(synapseOutput[name].sum())} spikes in {synapseOutput[name].shape[0]} fibers")

dt = method.dt
PSTHbinWidth = dt * 10                         # 10 samples per bin

psth = {}
for name in ["HSR", "MSR", "LSR"]:
    psth[name], dt = UTIL_PSTHmaker(synapseOutput[name].mean(axis=0, keepdims=True), dt, PSTHbinWidth)

txB = np.arange(psth["HSR"].shape[1]) * PSTHbinWidth

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(9, 7))
for ax, name in zip(axes, ["HSR", "MSR", "LSR"]):
    ax.plot(txB, psth[name][0] / PSTHbinWidth)
    ax.set_ylabel(f"{name}\n(spikes/s)")
axes[0].set_title("Post-stimulus time histograms")
axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

### Period histogram and vector strength
The spikes of all LSR fibers are folded on the stimulus period (0.5 ms for 2 kHz). The vector strength measures how
strongly the spikes are locked to a particular phase of the tone. If there are no spikes, it is `NaN`.
The original script also contains commented-out variants that use the MSR fibers or the PSTH instead.

In [ ]:
PH, binTimes = UTIL_periodHistogram(synapseOutput["LSR"].sum(axis=0, keepdims=True), dt, freq)

VS = UTIL_vectorStrength(PH)
print(f"Vector strength (LSR): {VS[0]:.4f}")

fig, ax = plt.subplots()
ax.plot(binTimes * 1e3, PH[0])
ax.set(xlabel="Time within period (ms)", ylabel="Spike count",
       title=f"Period histogram (LSR), VS = {VS[0]:.3f}")
plt.show()




## 7. What determines vector strength? AC/DC ratio of the receptor potential

Phase locking of an AN fiber starts with how strongly the IHC membrane potential follows the individual cycles of the
tone. A common measure of this is the **AC/DC ratio** of the receptor potential:

$$\text{AC/DC} = \frac{|V_{AC}|}{V_{DC} - V_{rest}}$$

- $|V_{AC}|$ is the amplitude of the component of *V* at the stimulus frequency (Fourier projection over a whole
  number of periods). Definitions differ in the literature (fundamental amplitude vs. peak-to-peak, a factor of 2).
- $V_{DC}$ is the mean of *V*. The DC *response* is the shift **relative to the resting potential**,
  $V_{DC}-V_{rest}$. Using the absolute mean of *V* (tens of mV below zero) would make the ratio meaningless.
- Both are measured in the **steady-state part of the tone** (here 30–105 ms, after the onset ramp and the first
  adaptation, before the offset ramp). The same window is used for the vector strength.

**Is it comparable with VS?** Only partly:

1. AC/DC of *V* describes the electrical stage only, and it is not bounded by 1. For example, a half-wave-rectified
   sinusoid has AC/DC = π/2 ≈ 1.57, while VS ≤ 1. The two can follow the same *trend* but not the same *values*.
2. VS has an exact counterpart for the *instantaneous rate* r(t) of the fiber: if r(t) = R₀ + A₁cos(ωt+φ) + (harmonics), then
   **VS = A₁ / (2R₀) = ½ · AC/DC of the rate**, where the DC is the mean rate itself (not relative to rest).
   This is the quantity that can be compared with VS one-to-one.
3. Between *V* and the rate sit calcium-channel activation (τM), calcium clearance (τCa), the expansive power law (exponent 3),
   vesicle depletion and the spontaneous rate (a larger mean rate lowers AC/DC). On top of this, the spikes add refractoriness
   and random sampling (finite spike counts).

The cells below sweep the tone frequency at **constant BM displacement**, which isolates the IHC and synapse stages
(in a real cochlea the BM amplitude would also change with frequency). For each frequency they compute:

- AC/DC of *V*,
- VS from the **spikes** of 200 fibers,
- VS from the deterministic **rate** (`ANprobability`: no randomness, no refractoriness), and a check that it equals ½ · AC/DC of that rate.

The frequencies are chosen so that `fs/freq` is an integer (needed by the period histogram).
The sweep runs the whole model 9 times and takes about a minute.

In [ ]:
types = ["HSR", "MSR", "LSR"]
win = (0.030, 0.105)          # steady-state window [s]: after the onset ramp (tone starts at 10 ms), before the offset ramp


def fundamental(x, fs, freq, win):
    """Mean (DC) and amplitude of the component at `freq` of x, using whole periods inside win = (t0, t1) [s]."""
    i0, i1 = (int(round(t * fs)) for t in win)
    seg = np.asarray(x)[i0:i1]
    nPeriods = int(np.floor(len(seg) * freq / fs))
    n = int(round(nPeriods * fs / freq))                       # whole number of periods
    seg = seg[:n]
    t = np.arange(n) / fs
    return seg.mean(), 2 * np.abs(np.mean(seg * np.exp(-2j * np.pi * freq * t)))


def ac_dc_ratio(x, fs, freq, win, rest=0.0):
    """AC/DC ratio: amplitude at `freq` divided by the DC shift (mean - rest)."""
    dc, ac = fundamental(x, fs, freq, win)
    return ac / (dc - rest)


def run_chain(freq, Amp, fs, rng, win):
    """Run stimulus -> IHC -> synapse -> AN for one tone and return the quantities compared below."""
    sig, _ = make_stimulus(freq, Amp, fs)
    m = SimpleNamespace(dt=1 / fs)
    V, m, _ = IHC_RP_Shamma(sig, m)
    rel, m = IHCpreSynapse(V.T, m)
    kt0_all = np.array(m.kt0)                                  # resting release rate of HSR / MSR / LSR

    i0, i1 = (int(round(t * fs)) for t in win)
    res = {"acdc_V": ac_dc_ratio(V[0], fs, freq, win, rest=m.restingV)}
    for k, name in enumerate(types):
        m.kt0 = kt0_all[k]
        spikes, m, rate = AN_IHCsynapse(rel[:, :, k].T, m, rng=rng)
        PH_spk, _ = UTIL_periodHistogram(spikes[:, i0:i1].sum(axis=0, keepdims=True), m.dt, freq)
        PH_rate, _ = UTIL_periodHistogram(rate[:, i0:i1], m.dt, freq)
        PH_all, _ = UTIL_periodHistogram(spikes.sum(axis=0, keepdims=True), m.dt, freq)
        res[name] = dict(
            VS_spikes=UTIL_vectorStrength(PH_spk)[0],              # from the spikes in the window
            VS_rate=UTIL_vectorStrength(PH_rate)[0],               # from the deterministic rate
            half_acdc_rate=0.5 * ac_dc_ratio(rate[0], fs, freq, win),   # should equal VS_rate
            mean_rate=spikes[:, i0:i1].mean() * fs,                # spikes/s per fiber in the window
            VS_spikes_whole=UTIL_vectorStrength(PH_all)[0],        # from the whole stimulus (incl. silence, ramps)
        )
    return res


freqs = [200,  500, 800, 1000, 1250, 2000, 2500, 4000, 5000, 6250, 10000]    # Hz; fs/freq must be an integer
rng_sweep = np.random.default_rng(SEED + 1)
results = [run_chain(f, Amp, fs, rng_sweep, win) for f in freqs]


acdc = np.array([r["acdc_V"] for r in results])
vs_spk = {n: np.array([r[n]["VS_spikes"] for r in results]) for n in types}
vs_rate = {n: np.array([r[n]["VS_rate"] for r in results]) for n in types}

# ---- table -------------------------------------------------------------------
print(f"Amp = {Amp * 1e9:.0f} nm, window {win[0] * 1e3:.0f}-{win[1] * 1e3:.0f} ms")
print(f"{'freq':>6} {'AC/DC(V)':>9} | {'VS from spikes':^21} | {'VS from rate':^21} | {'mean rate (spikes/s)':^23}")
print(f"{'(Hz)':>6} {'':>9} | " + " | ".join(" ".join(f"{t:>6}" for t in types) for _ in range(2))
      + " | " + " ".join(f"{t:>7}" for t in types))
for f, r in zip(freqs, results):
    print(f"{f:>6} {r['acdc_V']:>9.3f} | " + " | ".join(
        " ".join(f"{r[t][key]:>6.3f}" for t in types) for key in ("VS_spikes", "VS_rate"))
        + " | " + " ".join(f"{r[t]['mean_rate']:>7.1f}" for t in types))

err = max(abs(r[t]["VS_rate"] - r[t]["half_acdc_rate"]) for r in results for t in types)
print(f"\nmax |VS(rate) - 0.5 * AC/DC(rate)| over all runs = {err:.1e}")

i = freqs.index(freq) if freq in freqs else 0
print(f"\nVS from spikes at {freqs[i]} Hz, steady-state window vs. whole stimulus (incl. silence and ramps):")
for t in types:
    print(f"  {t}: {results[i][t]['VS_spikes']:.3f} vs {results[i][t]['VS_spikes_whole']:.3f}")

# ---- figure ------------------------------------------------------------------
colors = {"HSR": "#2a78d6", "MSR": "#eb6834", "LSR": "#1baf7a"}     # one fixed colour per fiber type
ink = "#52514e"
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

def plain_log_ticks(axis, ticks):
    """Readable tick labels (1000 instead of 10^3) on a log axis."""
    axis.set_ticks(ticks)
    axis.set_major_formatter(ScalarFormatter())
    axis.set_minor_formatter(NullFormatter())


ax = axes[0]
ax.loglog(freqs, acdc, "-o", color=ink, lw=1.5, ms=6, mec="white")
ax.set(xlabel="Frequency (Hz)", ylabel="AC/DC ratio of V", title="(a) Receptor potential")
plain_log_ticks(ax.xaxis, [250, 500, 1000, 2000, 5000, 10000])
plain_log_ticks(ax.yaxis, [0.1, 0.2, 0.5, 1, 2])

ax = axes[1]
iL = freqs.index(2000) if 2000 in freqs else 0               # place the direct labels where the curves are apart
for t in types:
    ax.semilogx(freqs, vs_rate[t], "-", color=colors[t], lw=1.5)
    ax.semilogx(freqs, vs_spk[t], "o", color=colors[t], ms=6, mec="white")
    ax.annotate(t, (freqs[iL], vs_rate[t][iL]), xytext=(8, 5), textcoords="offset points",
                color=ink, fontsize=9)
handles = [plt.Line2D([], [], color=colors[t], lw=1.5, label=t) for t in types] + [
    plt.Line2D([], [], color=ink, lw=1.5, label="rate (no noise, no refractoriness)"),
    plt.Line2D([], [], color=ink, marker="o", ls="", ms=6, mec="white", label="spikes (200 fibers)")]
ax.legend(handles=handles, fontsize=8, loc="upper right")
ax.set(xlabel="Frequency (Hz)", ylabel="Vector strength", title="(b) Vector strength", ylim=(0, 1))
plain_log_ticks(ax.xaxis, [250, 500, 1000, 2000, 5000, 10000])

ax = axes[2]
for t in types:
    ax.semilogx(acdc, vs_spk[t], "-o", color=colors[t], lw=1.5, ms=6, mec="white", label=t)
ax.annotate(f"{freqs[0]} Hz", (acdc[0], vs_spk["LSR"][0]), xytext=(6, 6), textcoords="offset points", color=ink, fontsize=9)
ax.annotate(f"{freqs[-1] / 1e3:g} kHz", (acdc[-1], vs_spk["LSR"][-1]), xytext=(6, 6), textcoords="offset points", color=ink, fontsize=9)
ax.legend(fontsize=8, loc="lower right")
ax.set(xlabel="AC/DC ratio of V", ylabel="Vector strength (spikes)", title="(c) VS vs. AC/DC of V", ylim=(0, 1))
plain_log_ticks(ax.xaxis, [0.1, 0.2, 0.5, 1, 2])

plt.tight_layout()
plt.show()

### What to look for (with the default settings)

- **Identity.** VS of the rate equals ½ · AC/DC of the rate to machine precision (printed check). It is an identity, not an approximation.
- **AC/DC of V is the same for HSR, MSR and LSR, but VS is not.** The three fiber types share *V* and differ only in the calcium
  clearance time constant τCa (see section 2). The ordering of VS at mid frequencies (LSR > MSR > HSR) therefore comes from the
  calcium stage, which acts as an additional low-pass filter between *V* and the release rate. So AC/DC of *V* alone does not determine VS (panel c).
- **Same trend, different values.** AC/DC of *V* falls by roughly a factor of 2 per octave at high frequencies (first-order low-pass behaviour of the
  membrane), and VS falls with it. At low frequencies AC/DC of *V* keeps growing beyond 1, while VS levels off below 1.
  Which of the stages between *V* and the rate causes this saturation? (Try changing parameters, or the level `Amp`.)
- **Spike VS vs. rate VS.** They agree where phase locking is strong. At high frequencies the VS from spikes levels off at about 0.02, whereas the VS from the
  rate keeps falling. This is the noise floor of a finite number of spikes: for *N* spikes with random phases the expected VS is about 0.89/√*N*
  (here *N* is roughly 2000, giving about 0.02). A VS of that size does not indicate phase locking.
- For the 2 kHz tone, the steady-state window used here gives practically the same VS as the whole stimulus used in section 6
  (the printed values differ by less than 0.01).

In [ ]:
# ---- Solution: stochastic vs. deterministic AN model, and refractoriness (HSR fibers) ----------------
name, nFib = "HSR", 2000
dt = 1 / fs

# Many independent fibers. The deterministic ("probability") output rate_det is returned by the same call.
params = dict(mode="spikes", numFibers=nFib, M=10, y=3, l=2580, x=30, r=6580,
              refractory_period=0.00075, returnSynapseContents=0)
method.kt0 = kt0[name]
spk, method, rate_det = AN_IHCsynapse(vesicle[name].T, method, params=params,
                                      rng=np.random.default_rng(SEED + 2))

# ---- 1) PSTH of N fibers vs. deterministic rate ------------------------------------------------------
binWidth = 1e-3                                              # 1 ms bins
nb = int(round(binWidth / dt))                               # samples per bin
det = UTIL_PSTHmaker(rate_det, dt, binWidth)[0][0] / nb      # deterministic rate averaged in each bin [spikes/s]
tb = np.arange(det.size) * binWidth                          # bin start times [s]
steady = (tb >= 0.030) & (tb < 0.105)                        # adapted part of the response


def psth_rate(group):
    """PSTH of a (fibers x time) spike matrix as the mean rate of one fiber [spikes/s]."""
    counts = UTIL_PSTHmaker(group.sum(axis=0, keepdims=True), dt, binWidth)[0][0]
    return counts / (group.shape[0] * binWidth)


# Split the fibers into independent groups of N; compare each group's PSTH with the deterministic rate
Ns = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000]
rmse_all, rmse_ss = [], []
for N in Ns:
    err = np.array([psth_rate(spk[g * N:(g + 1) * N]) - det for g in range(nFib // N)])   # groups x bins
    rmse_all.append(np.sqrt(np.mean(err ** 2)))              # whole response (incl. onset peak)
    rmse_ss.append(np.sqrt(np.mean(err[:, steady] ** 2)))    # steady-state part only
rmse_all, rmse_ss = np.array(rmse_all), np.array(rmse_ss)

# ---- 2) inter-spike intervals (ISI) in the adapted part of the response, vs. a Poisson process ------
def isi_ms(spikes, i0, i1):
    """Inter-spike intervals [ms] of all fibers (rows), using spikes in samples i0:i1."""
    return np.concatenate([np.diff(np.flatnonzero(row)) for row in spikes[:, i0:i1]]) * dt * 1e3


i0, i1 = (int(round(t * fs)) for t in (0.030, 0.105))       # steady-state window
rate_win = spk[:, i0:i1].mean() / dt                         # mean rate per fiber [spikes/s]
isi_model = isi_ms(spk, i0, i1)

# Poisson process with the same mean rate: spike with probability rate*dt in every sample, no memory
poisson = np.random.default_rng(SEED + 3).random((nFib, i1 - i0), dtype=np.float32) < rate_win * dt
isi_pois = isi_ms(poisson, 0, i1 - i0)

print(f"{name}: mean rate in the steady-state window: model {rate_win:.0f} spikes/s, deterministic {det[steady].mean():.0f} spikes/s")
print(f"onset peak: deterministic {det.max():.0f} spikes/s, PSTH of {nFib} fibers {psth_rate(spk).max():.0f} spikes/s\n")
print("RMS error of the PSTH [spikes/s]")
print(f"{'N':>6} {'steady state':>13} {'whole response':>15}")
for N, e1, e2 in zip(Ns, rmse_ss, rmse_all):
    print(f"{N:>6} {e1:>13.1f} {e2:>15.1f}")
print()
for label, isi in (("model", isi_model), ("Poisson", isi_pois)):
    print(f"{label:8s} shortest ISI = {isi.min():.2f} ms | ISIs < 1.5 ms: {100 * np.mean(isi < 1.5):4.1f} % | CV = {isi.std() / isi.mean():.2f}")

# ---- figure ---------------------------------------------------------------------------------------------
ink, blue, orange, aqua = "#52514e", "#2a78d6", "#eb6834", "#1baf7a"
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.plot(tb * 1e3, det, color=ink, lw=1.5, label="deterministic")
ax.plot(tb * 1e3, psth_rate(spk[:10]), color=blue, alpha=0.45, lw=1.2, label="PSTH, N = 10")
ax.plot(tb * 1e3, psth_rate(spk[:1000]), color=blue, lw=1.5, label="PSTH, N = 1000")
ax.set(xlabel="Time (ms)", ylabel="Rate (spikes/s)", title="(a) PSTH vs. deterministic rate")
ax.legend(fontsize=8)

ax = axes[1]
ax.loglog(Ns, rmse_ss, "-o", color=blue, lw=1.5, ms=6, mec="white", label="steady state (30-105 ms)")
ax.loglog(Ns, rmse_all, "-o", color=aqua, lw=1.5, ms=6, mec="white", label="whole response")
ax.loglog(Ns, rmse_ss[0] / np.sqrt(Ns), "--", color=ink, lw=1.2, label=r"$\propto 1/\sqrt{N}$")
ax.set_xticks([1, 10, 100, 1000])
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set(xlabel="Number of fibers N", ylabel="RMS error of PSTH (spikes/s)", title="(b) Error vs. number of fibers")
ax.legend(fontsize=8)

ax = axes[2]
binw = 0.2                                                   # ms
bins = np.arange(0, 20 + binw, binw)
for isi, color, label in ((isi_model, blue, "model (spikes)"), (isi_pois, orange, "Poisson, same rate")):
    # normalise by ALL intervals (also those > 20 ms), so that the histogram is comparable with the exponential density
    ax.hist(isi, bins, weights=np.full(isi.size, 1 / (isi.size * binw)), histtype="step", color=color, lw=1.5, label=label)
tt = np.linspace(0, 20, 400)
lam = rate_win * 1e-3                                        # rate per ms
ax.plot(tt, lam * np.exp(-lam * tt), "--", color=ink, lw=1.2, label="exponential")
ax.set(xlabel="Inter-spike interval (ms)", ylabel="Probability density (1/ms)", title="(c) ISI histogram")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()